
# Unit 2 — Advanced Visualization & Storytelling (UE24CS342AA9)
## Activity Notebook: Build an Interactive Retail Sales Dashboard

### Scenario

You have just joined the analytics team of **UrbanCart**, a mid-sized retail chain with stores across
three regions. Regional managers currently receive a static monthly PDF report and complain that:

1. They can't drill into *why* a number moved without emailing the analytics team.
2. Comparing stores or products side by side means flipping between many separate PDF pages.
3. Spotting a sudden dip in a specific product/store is slow and easy to miss.

**Your job:** apply the interaction techniques from the lecture — dynamic queries, coordinated views
& brushing, table lens, focus+context drill-down, and single-screen dashboard design — to build an
**interactive exploration tool** that solves these three complaints.

### How this notebook is organized

- A sample dataset is generated for you (Task 0) — don't skip running it.
- Each task states the **real-world usage**: which complaint above it solves, and why the technique
  fits.
- Tasks give you a **scaffold** (imports, helper stubs, expected output) — **you write the core logic**
  marked with `# TODO`. Do not just copy the Demo notebook's code verbatim — the data shape, column
  names and the exact question being asked are different here, so you will need to adapt the pattern,
  not paste it.


> **Grading-relevant tip:** every `# TODO` cell tells you what the final variable/plot must be named
> or show — read it before coding, since later cells depend on it.



## Task 0 — Load the Sample Dataset (provided)

Run the cells below as-is. This generates `sales_df`: 24 months of daily-aggregated sales across
6 stores (3 regions), 10 products (4 categories), including revenue, profit and customer ratings —
enough structure to support every technique you'll build below.


In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import interact, HBox, VBox

sns.set_theme(style="whitegrid")
np.random.seed(7)
print("Libraries loaded.")


Libraries loaded.


In [2]:

# --- Sample dataset generator: UrbanCart retail sales (provided, do not need to modify) ---
stores = pd.DataFrame({
    "store_id":   ["S01", "S02", "S03", "S04", "S05", "S06"],
    "store_city": ["Bengaluru", "Mysuru", "Mumbai", "Pune", "Delhi", "Jaipur"],
    "region":     ["South", "South", "West", "West", "North", "North"],
})

products = pd.DataFrame({
    "product":  ["Wireless Earbuds", "Smartwatch", "Laptop Sleeve", "USB-C Hub",
                 "Yoga Mat", "Dumbbell Set", "Running Shoes", "Track Jacket",
                 "Air Fryer", "Blender"],
    "category": ["Electronics", "Electronics", "Electronics", "Electronics",
                 "Fitness", "Fitness", "Apparel", "Apparel",
                 "Home", "Home"],
    "unit_price": [1499, 4999, 899, 1299, 799, 2499, 3499, 1999, 3999, 2299],
})

months = pd.date_range("2023-01-01", periods=24, freq="MS")

rows = []
for _, s in stores.iterrows():
    store_scale = np.random.uniform(0.7, 1.4)          # some stores just sell more
    for _, p in products.iterrows():
        product_scale = np.random.uniform(0.6, 1.6)
        trend = np.random.normal(0.01, 0.01)             # slow month-over-month growth/decline
        for i, m in enumerate(months):
            seasonal = 1 + 0.25 * np.sin(2 * np.pi * (m.month / 12))   # yearly seasonality
            units = max(0, np.random.poisson(15 * store_scale * product_scale * seasonal * (1 + trend) ** i))
            revenue = units * p["unit_price"]
            profit_margin = np.random.uniform(0.12, 0.35)
            profit = revenue * profit_margin
            rating = np.clip(np.random.normal(4.1, 0.4), 1, 5)
            # NOTE: use bracket indexing (p["product"]), not p.product -- pandas Series already has
            # a built-in .product() method, so attribute access on a column literally named "product"
            # silently returns that method instead of your data. This is a common real-world gotcha.
            rows.append((m, s["store_id"], s["store_city"], s["region"], p["product"], p["category"],
                         p["unit_price"], units, revenue, profit, round(rating, 1)))

sales_df = pd.DataFrame(rows, columns=[
    "month", "store_id", "store_city", "region", "product", "category",
    "unit_price", "units_sold", "revenue", "profit", "avg_rating"
])
for _col in ["units_sold", "revenue", "profit"]:
    sales_df[_col] = sales_df[_col].astype(float)  # float, so the anomaly injection below can scale them

# Inject one deliberate "anomaly" for Task 4 (a real dip a manager would want to investigate)
mask = (sales_df.store_id == "S03") & (sales_df["product"] == "Smartwatch") & (sales_df.month == "2024-06-01")
sales_df.loc[mask, ["units_sold", "revenue", "profit"]] = sales_df.loc[mask, ["units_sold", "revenue", "profit"]] * 0.15

print(sales_df.shape)
sales_df.head()


(1440, 11)


[styled table output - stripped to keep file size small; re-run cell to view]


**Reflection (answer in a sentence or two, in this markdown cell):**
Before writing any code, look at the columns above. Which columns would you use as *filters* for
dynamic querying, and which would you use to *link* views together for brushing? Write your answer
here.

_Your answer:_ Good filter columns are the numeric ranges managers reason about — `unit_price`,
`units_sold`, `revenue`, `profit`, `avg_rating` — since dynamic-query sliders need an ordered range.
`store_id`/`store_city`/`region` and `product`/`category` are the categorical link keys: selecting
one in an overview view and using it to filter/highlight the same rows in other views is exactly
what brushing does.


## Task 1 — Static vs. Interactive: Which Report Actually Answers the Manager's Question?

**Real-world usage:** Complaint #1 ("flipping through PDF pages"). A regional manager asks:
*"Show me total revenue by product, and let me check exactly how much any bar is worth without
squinting at a printed axis."*

**Why this technique:** A static bar chart is fine for the *headline* number, but it can't answer a
precise follow-up question live — that needs a hover-enabled interactive chart (Slide 4-8).

### Your task
1. Build a **static** `matplotlib`/`seaborn` bar chart of total revenue per `product`, sorted
   descending.
2. Build the **same** chart as an **interactive** `plotly` bar chart where hovering shows the exact
   revenue figure and the category of that product.
3. In the markdown cell below, state in 1-2 sentences: for *this specific* manager request, which
   version wins, and why (tie your answer to target-audience / data-story / ROI from the slide).


In [3]:
# --- 1a. STATIC chart ---
revenue_by_product = sales_df.groupby("product")["revenue"].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=revenue_by_product.values, y=revenue_by_product.index, color="steelblue", ax=ax)
ax.set_xlabel("Total Revenue ($)")
ax.set_ylabel("Product")
ax.set_title("STATIC: Total Revenue by Product (sorted descending)")
plt.tight_layout()
plt.show()


[plot rendered - image stripped to keep file size small; re-run cell to view]


In [4]:
# --- 1b. INTERACTIVE chart ---
rev_cat = sales_df.groupby(["product", "category"], as_index=False)["revenue"].sum()
rev_cat = rev_cat.sort_values("revenue", ascending=False)

fig = px.bar(
    rev_cat, x="product", y="revenue", color="category",
    hover_data={"revenue": ":,.0f", "category": True},
    title="INTERACTIVE: Total Revenue by Product (hover for exact revenue and category)"
)
fig.update_layout(xaxis_tickangle=45, height=480)
fig.show()


**Your 1-2 sentence answer:**

_Type here._ The interactive Plotly version wins for this request: the manager explicitly asked to
"check exactly how much any bar is worth without squinting," which is a precise, per-item lookup —
exactly what hover-enabled dynamic queries are for, whereas the static chart only supports the
headline take-away, not a live follow-up question.


## Task 2 — Dynamic Queries: Let Managers Filter Without Asking You

**Real-world usage:** Regional managers want to self-serve answer questions like *"which
products, in what price range, sold above X units last month?"* without emailing the analytics team
every time (Complaint #1 again, but for filtering rather than just reading).

**Why this technique:** Dynamic queries (double-ended range sliders) let the pattern emerge from the
noise live, with the response updating in real time (Slide 23).

### Your task
Build **two** linked `ipywidgets` range sliders:
- `price_slider` — filters on `unit_price`
- `units_slider` — filters on `units_sold`

Write a function `dynamic_filter(price_range, units_range)` that:
1. Filters `sales_df` to rows where `unit_price` is inside `price_range` **and** `units_sold` is
   inside `units_range`.
2. Prints how many rows match out of the total.
3. Shows a scatter plot of `unit_price` vs `units_sold`, with matching rows highlighted in a
   distinct color against all other rows in gray (same pattern as the Demo notebook's dynamic query
   cell — but you must build the filter condition and the plot yourself for these two columns).

Wire it up with `interact()`.


In [5]:
price_slider = widgets.FloatRangeSlider(
    value=[sales_df.unit_price.min(), sales_df.unit_price.max()],
    min=sales_df.unit_price.min(), max=sales_df.unit_price.max(), step=50,
    description="Price range:", continuous_update=True,
    layout=widgets.Layout(width="500px")
)

units_slider = widgets.FloatRangeSlider(
    value=[sales_df.units_sold.min(), sales_df.units_sold.max()],
    min=sales_df.units_sold.min(), max=sales_df.units_sold.max(), step=1,
    description="Units range:", continuous_update=True,
    layout=widgets.Layout(width="500px")
)

def dynamic_filter(price_range, units_range):
    filtered = sales_df[
        sales_df.unit_price.between(*price_range) & sales_df.units_sold.between(*units_range)
    ]
    print(f"{len(filtered)} / {len(sales_df)} rows match")

    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.scatter(sales_df.unit_price, sales_df.units_sold, s=10, c="lightgray")
    ax.scatter(filtered.unit_price, filtered.units_sold, s=25, c="crimson")
    ax.set_xlabel("Unit Price"); ax.set_ylabel("Units Sold")
    ax.set_title(f"Dynamic Query: {len(filtered)}/{len(sales_df)} rows match current slider ranges")
    plt.tight_layout()
    plt.show()

interact(dynamic_filter, price_range=price_slider, units_range=units_slider)


# Static render for documentation (in addition to the interactive widget above)
dynamic_filter(price_slider.value, units_slider.value)


1440 / 1440 rows match


[plot rendered - image stripped to keep file size small; re-run cell to view]


**Reflection:** The slide notes dynamic queries work well "up to ~100,000 points with five or fewer
sliders." `sales_df` has far fewer rows. If UrbanCart grew to 500 stores and this dataset became 50x
larger, would two sliders still be enough, or would you need a different technique from the lecture?
Name one.

_Your answer:_ Two sliders would likely still stay under the ~100,000-point ceiling if the growth
mostly increases store count, since the filtering logic itself doesn't get slower — but rendering a
scatter of that many raw points would get visually cluttered. At that scale I'd pair dynamic queries
with a Table Lens or aggregate the scatter (e.g. hexbin/density) so the query still runs fast but the
result stays readable rather than an overplotted blob.


## Task 3 — Coordinated Views & Cross-View Brushing: Compare Stores Without Flipping Pages

**Real-world usage:** Complaint #2 — "comparing stores means flipping between many PDF pages."
A manager wants to click one store in an overview chart and instantly see that store's category
breakdown and its monthly trend, without losing sight of how it compares to the others.

**Why this technique:** Coordinated multiple views + brushing (Slide 24) — selecting a subset in one
view highlights the same records everywhere else, replacing the working-memory cost of flipping pages.

### Your task
Build **three linked views** for a store the user selects from a dropdown:
1. **Overview bar chart** — total revenue per store (all 6 stores), with the selected store's bar in
   a distinct color and every other store's bar in gray (this is the "muted unselected categories"
   trick from the slide).
2. **Category breakdown** — a pie or bar chart of revenue by `category`, filtered to the selected
   store only.
3. **Monthly trend line** — revenue by `month`, filtered to the selected store only.

Wrap all three in one function driven by a single `ipywidgets.Dropdown` of `store_id` values, and lay
the three charts out together (e.g. with `plt.subplots` for 1+2+3, or separate `display()` calls).


In [6]:
store_overview = sales_df.groupby("store_id").revenue.sum().reset_index()

def store_coordinated_view(selected_store):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    # 1. Overview bar chart, selected store highlighted
    colors = ["crimson" if s == selected_store else "lightgray" for s in store_overview.store_id]
    axes[0].bar(store_overview.store_id, store_overview.revenue, color=colors)
    axes[0].set_title(f"Overview: Revenue by Store\n(highlighted = {selected_store})")
    axes[0].set_ylabel("Total Revenue ($)")

    # 2. Category breakdown for selected store
    cat_rev = sales_df[sales_df.store_id == selected_store].groupby("category").revenue.sum()
    axes[1].pie(cat_rev.values, labels=cat_rev.index, autopct="%1.0f%%", startangle=90)
    axes[1].set_title(f"Category Breakdown: {selected_store}")

    # 3. Monthly trend for selected store
    trend = sales_df[sales_df.store_id == selected_store].groupby("month").revenue.sum()
    axes[2].plot(trend.index, trend.values, marker="o", color="crimson")
    axes[2].set_title(f"Monthly Revenue Trend: {selected_store}")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()

interact(store_coordinated_view, selected_store=widgets.Dropdown(options=list(stores["store_id"]), description="Store:"))


# Static render for documentation (in addition to the interactive widget above)
store_coordinated_view("S03")


[plot rendered - image stripped to keep file size small; re-run cell to view]


**Reflection:** Which of the three linked views above is playing the role of "context" and which is
playing "focus," in the sense of the Focus-Context Problem (Slide 18)? Is that the same distinction as
brushing, or a different but related idea? Explain briefly.

_Your answer:_ The overview bar chart is the "context" view — it keeps every store visible so the
selected one is seen relative to the whole chain. The category breakdown and monthly trend are the
"focus" views — they zoom into detail for just the selected store. This is related to brushing (both
rely on one selection propagating everywhere) but is a distinct idea: brushing is about *linking*
selections across views, while focus+context is about keeping the *overview visible at the same time*
as the detail, rather than replacing it.


## Task 4 — Focus + Context Drill-Down: Investigate the Dip

**Real-world usage:** Complaint #3 — "spotting a sudden dip is slow and easy to miss." One product at
one store has a real anomaly hidden in `sales_df` (planted in Task 0). A manager needs to find it and
drill in **without losing the overview**, using the *cheapest* interaction tier that still answers the
question (Drill-Down Cost Hierarchy, Slide 17).

### Your task
**Part A — find the anomaly.**
Write code that computes, for every `(store_id, product)` pair, the ratio of the *minimum* month's
`units_sold` to the *median* month's `units_sold` across the 24 months. Sort ascending and show the
top 5 most extreme drops. (Hint: `groupby(["store_id","product"]).units_sold.agg(...)` with a custom
function, or compute `min` and `median` separately and combine.)

**Part B — build the drill-down.**
Using **hover** (cheap tier) on an overview chart of all `(store_id, product)` trend lines, let the
user identify the anomalous line. Then, given the `store_id`/`product` you found in Part A, plot its
monthly trend **alongside** a faint gray context line showing the *average* trend across all
stores/products for the same product — so the dip is visible with its context still present (not a
full window replacement).


In [7]:
# --- Part A: find the anomaly ---
def min_median_ratio(s):
    med = s.median()
    return s.min() / med if med != 0 else np.nan

anomaly_ranking = (
    sales_df.groupby(["store_id", "product"]).units_sold
    .agg(min_median_ratio)
    .sort_values()
    .head(5)
)
print(anomaly_ranking)


store_id  product     
S03       Smartwatch      0.185714
S05       USB-C Hub       0.250000
S01       Track Jacket    0.300000
S03       USB-C Hub       0.303030
S02       Yoga Mat        0.315789
Name: units_sold, dtype: float64


In [8]:
# --- Part B: drill-down with context preserved ---
# TODO 1: overview line chart, one line per (store_id, product)
overview_line = px.line(
    sales_df.sort_values("month"), x="month", y="units_sold",
    color="store_id", line_group="product",
    hover_data=["store_id", "product"],
    title="OVERVIEW (cheap tier — hover): units_sold trend per (store_id, product)"
)
overview_line.update_traces(line=dict(width=1), opacity=0.4)
overview_line.update_layout(height=450, showlegend=False)
overview_line.show()

# TODO 2: focus (top anomaly) + context (product-wide average)
top_store, top_product = anomaly_ranking.index[0]

focus = sales_df[(sales_df.store_id == top_store) & (sales_df.product == top_product)] \
    .sort_values("month")
context = sales_df[sales_df["product"] == top_product].groupby("month").units_sold.mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(context.index, context.values, color="gray", alpha=0.6,
        label=f"Context: avg '{top_product}' trend, all stores")
ax.plot(focus.month, focus.units_sold, color="crimson", marker="o",
        label=f"Focus: {top_store} / {top_product}")
ax.set_title(f"Focus + Context Drill-Down: {top_store} / {top_product} vs. product-wide average")
ax.set_xlabel("Month"); ax.set_ylabel("Units Sold")
ax.legend()
plt.tight_layout()
plt.show()


[plot rendered - image stripped to keep file size small; re-run cell to view]


**Reflection:** Which interaction-cost tier (eye fixation / hover / click-to-open panel / window
replacement) did you use to first *spot* the anomaly, and which did you use to *investigate* it? Was
using two different tiers the right call, per the lecture's design rule?

_Your answer:_ Spotting used the cheap hover tier on the overview chart — no navigation was needed,
just moving the cursor over lines. Investigating used a click-to-open-style panel (the dedicated
focus+context chart built from the Part A ranking). Using two different tiers is exactly the design
rule: use the cheapest tier that answers each question, escalating to a more expensive tier only once
the cheap tier can no longer resolve the detail needed.


## Task 5 — Table Lens: A Scannable Product Performance Table

**Real-world usage:** A manager wants a single table listing every product's total revenue, total
profit, and average rating across all stores — but a plain spreadsheet of 10 rows x many numeric
columns is still hard to compare at a glance across all three metrics simultaneously.

**Why this technique:** Table Lens (Slide 27) maps numeric columns to in-cell bars, so magnitude
differences are visible pre-attentively while the table stays sortable/readable as text.

### Your task
1. Build a summary DataFrame: one row per `product`, columns for total `revenue`, total `profit`, and
   mean `avg_rating`.
2. Sort it by total revenue, descending.
3. Style it as a Table Lens using `.style.bar(...)` — a different bar color per numeric column, as in
   the Demo notebook — and add a meaningful `.set_caption(...)`.
4. Below the table, add **one more cell**: sort the *same* summary table by `profit` instead of
   `revenue`, and re-render the styled table. Do the top rows change order? Note what that tells a
   manager about revenue vs. profitability.


In [9]:
product_summary = sales_df.groupby("product", as_index=False).agg(
    revenue=("revenue", "sum"),
    profit=("profit", "sum"),
    avg_rating=("avg_rating", "mean"),
)
product_summary = product_summary.sort_values("revenue", ascending=False).reset_index(drop=True)

styled = (
    product_summary.style
    .bar(subset=["revenue"], color="#5DADE2")
    .bar(subset=["profit"], color="#58D68D")
    .bar(subset=["avg_rating"], color="#F5B041")
    .format({"revenue": "{:,.0f}", "profit": "{:,.0f}", "avg_rating": "{:.2f}"})
    .set_caption("Table Lens: product performance, sorted by revenue — bars show magnitude at a glance")
)
styled


[styled table output - stripped to keep file size small; re-run cell to view]


In [10]:
product_summary_by_profit = product_summary.sort_values("profit", ascending=False).reset_index(drop=True)

styled_profit = (
    product_summary_by_profit.style
    .bar(subset=["revenue"], color="#5DADE2")
    .bar(subset=["profit"], color="#58D68D")
    .bar(subset=["avg_rating"], color="#F5B041")
    .format({"revenue": "{:,.0f}", "profit": "{:,.0f}", "avg_rating": "{:.2f}"})
    .set_caption("Table Lens: same products, re-sorted by profit — compare top rows to the revenue sort")
)
styled_profit


[styled table output - stripped to keep file size small; re-run cell to view]


**Your observation (revenue-sorted vs profit-sorted top rows):**

_Type here._ The top rows shift order once sorted by profit instead of revenue — a product with high
revenue but a thinner margin can drop several ranks while a lower-revenue, higher-margin product rises.
This tells a manager that chasing top-line revenue alone can mask which products actually drive
profitability, and the two rankings should be checked together before deciding what to promote.


## Task 6 — Assemble a Single-Screen Monitoring Dashboard (open-ended)

**Real-world usage:** All three complaints at once. UrbanCart wants a **single screen** a regional
manager can glance at every morning — no interaction required to read the headline state, per the
Dashboard Patterns slide (Slide 28): critical indicators visible at a glance, with a peripheral-tier
alert that uses color (not subtlety) to flag something that needs attention.

### Your task
Build a 2x2 dashboard (`plt.subplots(2, 2, ...)`) with:

1. **Top-left — KPI summary panel** (`ax.axis("off")` + `ax.text(...)`): total revenue, total profit,
   overall average rating, and number of active stores, computed from `sales_df`.
2. **Top-right — mid-term analysis panel**: revenue by `region` (bar chart).
3. **Bottom-left — long-term trend panel**: total revenue by `month`, across all stores (line chart).
4. **Bottom-right — peripheral alert panel**: check whether **any** `(store_id, product)` pair has a
   month where `units_sold` fell below **25% of its own median** for that pair (this is the anomaly
   you found in Task 4, but write the check generically so it would catch *any* such case, not just
   the one you already know about). Show a red "ALERT — N issue(s) found" indicator if any exist, or
   a green "OK" indicator if not.

This task is intentionally the least scaffolded — you decide the exact `groupby` calls, panel layout
details, and styling. Reuse patterns from Tasks 1-5 and the Demo notebook where they fit, but write
the logic yourself.


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# --- Panel 1: KPI summary (top-left) ---
axes[0, 0].axis("off")
kpi_text = (
    f"Total revenue: ${sales_df.revenue.sum():,.0f}\n"
    f"Total profit: ${sales_df.profit.sum():,.0f}\n"
    f"Avg rating: {sales_df.avg_rating.mean():.2f}\n"
    f"Active stores: {sales_df.store_id.nunique()}"
)
axes[0, 0].text(0.05, 0.5, kpi_text, fontsize=13, va="center")
axes[0, 0].set_title("KPI Summary")

# --- Panel 2: revenue by region (top-right) ---
region_rev = sales_df.groupby("region").revenue.sum().sort_values(ascending=False)
axes[0, 1].bar(region_rev.index, region_rev.values, color="teal")
axes[0, 1].set_title("Revenue by Region")

# --- Panel 3: total revenue by month (bottom-left) ---
month_rev = sales_df.groupby("month").revenue.sum()
axes[1, 0].plot(month_rev.index, month_rev.values, color="navy")
axes[1, 0].set_title("Total Revenue by Month")
axes[1, 0].tick_params(axis="x", rotation=45)

# --- Panel 4: peripheral alert panel (bottom-right) ---
def ratio(s):
    med = s.median()
    return s.min() / med if med != 0 else np.nan

ratios = sales_df.groupby(["store_id", "product"]).units_sold.agg(ratio)
n_alerts = int((ratios < 0.25).sum())

axes[1, 1].axis("off")
alert_color = "red" if n_alerts > 0 else "green"
axes[1, 1].add_patch(mpatches.Circle((0.5, 0.5), 0.35, color=alert_color, alpha=0.85))
label = f"ALERT — {n_alerts} issue(s) found" if n_alerts > 0 else "OK"
axes[1, 1].text(0.5, 0.5, label, ha="center", va="center", color="white", fontsize=13, weight="bold")
axes[1, 1].set_title("Anomaly Alert (< 25% of own median in any month)")

fig.suptitle("UrbanCart Regional Manager Dashboard", fontsize=14)
plt.tight_layout()
plt.show()


[plot rendered - image stripped to keep file size small; re-run cell to view]


## Reflection

Answer briefly, referring back to the three manager complaints from the scenario intro:

1. Which single technique from this activity do you think gives UrbanCart's managers the *biggest*
time saving, and why?
2. Which technique was hardest to implement well, and what design trade-off (from the lecture) did you
have to think about while building it?
3. If you had to add **one more** interaction technique from the lecture that this activity didn't
cover (e.g. geometric fisheye, magic lens, network zooming, model-based planning), where in this
retail scenario would it actually be useful, and why?

_Your answers here._

1. The single-screen dashboard (Task 6) likely saves the most time overall, since it directly answers
all three complaints at once every morning with zero interaction required — managers no longer wait
on emails, flip pages, or hunt for dips.
2. The coordinated-views brushing in Task 3 was hardest, because it required deciding how much
context to keep visible (all six stores muted gray) versus how much detail to show for the
selection — too much muted context clutters the screen, too little loses the comparison the manager
asked for.
3. Model-based planning (a growth-rate slider projecting future revenue with an uncertainty band)
would be useful for the monthly trend panel — it would let a manager ask "what if this dip continues"
rather than only seeing what already happened.
